# Notebook 2 — a viga engastada

Minicurso **Da kalimba à viga** · Bloco 2

Três tarefas: achar as raízes da equação característica, desenhar as formas modais com os
nós, e comparar a frequência calculada da sua lamela com o espectro medido.

## 1. As raízes de $\cos(\beta L)\cosh(\beta L) + 1 = 0$

A partir da segunda, as raízes ficam muito perto de $(n-\tfrac12)\pi$, porque
$\cosh\beta L$ cresce tão depressa que a equação vira, na prática, $\cos\beta L = 0$.

In [ ]:
import numpy as np
from scipy.optimize import brentq

def caracteristica(bL):
    """cos(bL) cosh(bL) + 1 = 0, a equacao da viga engastada-livre."""
    return np.cos(bL) * np.cosh(bL) + 1.0

chutes = [(1.0, 2.5), (4.0, 5.5), (7.0, 8.5), (10.0, 11.5)]
raizes = np.array([brentq(caracteristica, a, b) for a, b in chutes])

print("beta_n L :", np.round(raizes, 4))
print("f_n/f_1  :", np.round((raizes/raizes[0])**2, 3))

Saem `1.8751 4.6941 7.8548 10.9955` e as razões `1 6.267 17.55 34.386`.

Compare com a corda, onde as razões seriam 1, 2, 3, 4. É essa diferença que o ouvido
escuta.

**Cuidado numérico.** Para raízes mais altas, `cosh` estoura. Use a forma equivalente
abaixo, que é bem mais bem comportada.

In [ ]:
def caracteristica_estavel(bL):
    """Equivalente a cos(bL)cosh(bL)+1=0, dividida por cosh(bL)."""
    return np.cos(bL) + 1.0/np.cosh(bL)

altas = [brentq(caracteristica_estavel, (n - 0.5)*np.pi - 0.4,
                (n - 0.5)*np.pi + 0.4) for n in range(2, 11)]
print(np.round(altas, 4))

## 2. As frequências

In [ ]:
import numpy as np

def frequencias_viga(L, h, b, E, rho, raizes):
    """Frequencias naturais [Hz] de uma viga engastada-livre de secao retangular."""
    I = b * h**3 / 12
    A = b * h
    return (raizes**2) / (2*np.pi*L**2) * np.sqrt(E*I / (rho*A))

# lamela de aco: 40 mm livres, 1 mm de espessura, 10 mm de largura
L, h, b = 0.040, 0.001, 0.010
E, rho = 200e9, 7850

f = frequencias_viga(L, h, b, E, rho, raizes)
print(np.round(f, 1))        # [ 509.6 3193.7 8942.4 17523.6]

**Confira as leituras da fórmula.** Mude `b` e veja que nada acontece; dobre `h`
e a frequência dobra; dobre `L` e ela cai por quatro.

In [ ]:
for nome, args in [("original  ", (L, h, b)),
                   ("b dobrado ", (L, h, 2*b)),
                   ("h dobrado ", (L, 2*h, b)),
                   ("L dobrado ", (2*L, h, b))]:
    print(nome, round(frequencias_viga(*args, E, rho, raizes)[0], 1), "Hz")

## 3. As formas modais

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def forma(x, bL):
    """Forma modal da viga engastada-livre, x em unidades de L."""
    s = (np.cos(bL) + np.cosh(bL)) / (np.sin(bL) + np.sinh(bL))
    return (np.cosh(bL*x) - np.cos(bL*x)
            - s * (np.sinh(bL*x) - np.sin(bL*x)))

x = np.linspace(0, 1, 400)
plt.figure(figsize=(9, 4.5))
for n, bL in enumerate(raizes[:3], start=1):
    y = forma(x, bL)
    y = y / np.max(np.abs(y))
    plt.plot(x, y, label=f"modo {n}")
    troca = np.where(np.sign(y[:-1]) != np.sign(y[1:]))[0]   # nos
    plt.plot(x[troca], y[troca], "o", color="firebrick")

plt.axhline(0, color="0.8", lw=1)
plt.xlabel("x / L"); plt.ylabel("amplitude normalizada")
plt.legend(); plt.tight_layout(); plt.show()

## 4. A sua lamela

Meça com o paquímetro, preencha abaixo e compare com o espectro medido no notebook 1.

Se o medido ficar **abaixo** do calculado, isso é esperado: o engaste real cede um pouco,
o que equivale a um comprimento livre maior.

In [ ]:
# preencha com as suas medidas
L_medido = 0.040      # m
h_medido = 0.001      # m
b_medido = 0.010      # m
material = ("aco", 200e9, 7850)

f_calc = frequencias_viga(L_medido, h_medido, b_medido, material[1], material[2], raizes)
print("f1 calculado:", round(f_calc[0], 1), "Hz")

f_medido = None       # ponha aqui o pico do espectro
if f_medido:
    print("f1 medido   :", f_medido, "Hz")
    print("diferenca   :", round(100*(f_medido - f_calc[0])/f_calc[0], 1), "%")
    L_efetivo = L_medido * np.sqrt(f_calc[0]/f_medido)
    print("L efetivo   :", round(1000*L_efetivo, 1), "mm  (contra",
          round(1000*L_medido, 1), "medidos)")